# GATHeR — сборка BCR из симулированных PE150 — мышь ERP003950

Шесть образцов, только тяжёлая цепь IgG (IGH); каждый образец обрабатывается отдельно. Используется та же базовая ветка симуляции, что и в `assemble_trust4_mouse.ipynb`.

GATHeR 1.0.1 строит сжатый граф де Брёйна из ридов, в каждой компоненте связности выбирает путь с максимальным средним весом, добавляет контиги rnaSPAdes и аннотирует контиги BLASTN по базе IMGT. Метод рассчитан на отдельные клетки (Smart-seq, 10x); здесь образец целиком подаётся как один вход `sc-asm`.

Параметры: база мыши, `k=30` (значение по умолчанию установленной версии), `min_freq=5`. rnaSPAdes вызывается через обёртку `scripts/gather_spades_compat.sh`: если `transcripts.fasta` не создан, берутся финальные контиги последнего k. Выход для каждого образца: `<ветка>/assemblers/gather/<образец>/` — нативные FASTA в `native/` (unitigs, contigs, BCR), `run.log`, `run_metadata.json` и нормализованный `contigs.fasta` для сравнения с эталоном; в его заголовке сохраняются аннотация, имя алгоритма и вес `W` из нативного файла. Корень вывода задаётся переменной `BCR_GATHER_OUT`. Образец с готовым `run_metadata.json` не пересчитывается (`BCR_FORCE=1` — пересчитать).


## Окружение и параметры

In [ ]:
import os, shutil, subprocess, time, gzip, json
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

BCR_ENV = Path(os.environ.get("BCR_ENV", "/data/user/epishkin/conda/envs/bcr_env"))
GATHER_RUNTIME = Path(os.environ.get("BCR_GATHER_RUNTIME", "/data/user/epishkin/conda/envs/gather_runtime"))
os.environ["PATH"] = str(BCR_ENV / "bin") + os.pathsep + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(filter(None, [str(BCR_ENV / "lib"), str(GATHER_RUNTIME / "lib"), os.environ.get("LD_LIBRARY_PATH", "")]))

DATASET = "ERP003950"
BRANCH = os.environ.get("BCR_BRANCH", "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut_amp_ec1x3_in5ng_rb3x_ss250")
SAMPLES = os.environ["BCR_SAMPLES"].split(",") if os.environ.get("BCR_SAMPLES") else ["ERR346596", "ERR346597", "ERR346598", "ERR346599", "ERR346600", "ERR346601"]
THREADS = int(os.environ.get("BCR_THREADS", 20))
GATHER_K = int(os.environ.get("BCR_GATHER_K", 30))
MIN_FREQ = int(os.environ.get("BCR_GATHER_MIN_FREQ", 5))
FORCE = os.environ.get("BCR_FORCE", "0") == "1"

def find_root():
    cwd = Path.cwd().resolve()
    for root in (Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin")), cwd, *cwd.parents):
        if (root / "results" / DATASET / "simulated" / BRANCH).is_dir():
            return root
    raise FileNotFoundError(f"results/{DATASET}/simulated/{BRANCH} not found; set BCR_VOLUME")

ROOT = find_root()
BRANCH_DIR = ROOT / "results" / DATASET / "simulated" / BRANCH
FASTQ_DIR = BRANCH_DIR / "06_fastq_pe150"
GATHER_DIR = Path(os.environ.get("BCR_GATHER_OUT", BRANCH_DIR / "assemblers" / "gather"))
BLAST_DIR = Path(os.environ.get("BCR_BLAST_DIR", GATHER_RUNTIME / "bin"))
_compat = ROOT / "scripts" / "gather_spades_compat.sh"
SPADES = Path(os.environ.get("BCR_SPADES_PATH", _compat if _compat.exists() else GATHER_RUNTIME / "bin" / "spades.py"))

def reads(sample):
    r1, r2 = FASTQ_DIR / f"{sample}_R1.fastq.gz", FASTQ_DIR / f"{sample}_R2.fastq.gz"
    if not (r1.exists() and r2.exists()):
        raise FileNotFoundError(f"simulated FASTQ missing for {sample}: {r1}")
    return r1, r2

def run_logged(cmd, log_path, cwd=None, heartbeat=300):
    log_path = Path(log_path); log_path.parent.mkdir(parents=True, exist_ok=True)
    print("RUN:", " ".join(map(str, cmd)), "\nLOG:", log_path, flush=True)
    t0, last = time.time(), time.time()
    with open(log_path, "w") as log:
        p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
        while p.poll() is None:
            time.sleep(10)
            if time.time() - last >= heartbeat:
                print(f"  running {(time.time() - t0) / 60:.0f} min", flush=True); last = time.time()
    elapsed = time.time() - t0
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}; see {log_path}")
    print(f"done in {elapsed / 60:.1f} min", flush=True)
    return elapsed

def iter_fasta(path, full_header=False):
    name, chunks = None, []
    with (gzip.open(path, "rt") if str(path).endswith(".gz") else open(path)) as h:
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = (line[1:].strip() if full_header else line[1:].split()[0]), []
            else:
                chunks.append(line.strip())
    if name is not None:
        yield name, "".join(chunks)

def write_contigs(src_pairs, out_fa, prefix):
    n = 0
    tmp = Path(str(out_fa) + ".tmp")
    with open(tmp, "w") as h:
        for name, seq in src_pairs:
            seq = seq.upper()
            if seq:
                n += 1
                h.write(f">{prefix}_{n} {name}\n{seq}\n")
    tmp.replace(out_fa)
    print(out_fa, n, "contigs")
    return n

required = [BCR_ENV / "bin" / x for x in ("sc-asm", "bcalm")] + [BLAST_DIR / "blastn", BLAST_DIR / "makeblastdb", SPADES]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f"missing GATHeR runtime dependencies: {missing}")

print("ROOT:", ROOT)
print("BRANCH:", BRANCH_DIR)
print("samples:", SAMPLES, "threads:", THREADS)
print("GATHeR:", version("gather"), "k:", GATHER_K, "min_freq:", MIN_FREQ)
print(subprocess.run([str(BLAST_DIR / "blastn"), "-version"], capture_output=True, text=True, check=True).stdout.splitlines()[0])
print(subprocess.run([str(SPADES), "--version"], capture_output=True, text=True, check=True).stdout.strip())

## Запуск GATHeR

In [ ]:
for sample in SAMPLES:
    out = GATHER_DIR / sample
    contigs = out / "contigs.fasta"
    if (out / "run_metadata.json").exists() and not FORCE:
        print("[skip]", sample)
        continue

    r1, r2 = reads(sample)
    out.mkdir(parents=True, exist_ok=True)
    stage = out / ".native.staging"
    native = out / "native"
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir()

    cmd = [BCR_ENV / "bin" / "sc-asm",
           "--seq_1", r1, "--seq_2", r2,
           "--k", GATHER_K, "--min_freq", MIN_FREQ,
           "--output_dir", stage, "--num_jobs", THREADS,
           "--mouse", "--blast_dir", BLAST_DIR, "--spades_path", SPADES]
    elapsed = run_logged(cmd, out / "run.log")

    bcr_outputs = sorted(stage.glob("*BCR.fa"))
    if len(bcr_outputs) != 1:
        raise RuntimeError(f"expected one GATHeR BCR FASTA for {sample}, found {bcr_outputs}")
    bcr_name = bcr_outputs[0].name
    if native.exists():
        shutil.rmtree(native)
    stage.replace(native)
    native_bcr = native / bcr_name
    n = write_contigs(iter_fasta(native_bcr, full_header=True), contigs, "gather")

    metadata = {
        "sample": sample, "dataset": DATASET, "branch": BRANCH,
        "gather_version": version("gather"), "k": GATHER_K,
        "min_freq": MIN_FREQ, "threads": THREADS,
        "bulk_as_single_input": True, "contigs": n,
        "native_bcr_fasta": str(native_bcr.relative_to(out)), "elapsed_seconds": round(elapsed, 1),
        "finished_utc": datetime.now(timezone.utc).isoformat(),
        "command": list(map(str, cmd)),
    }
    (out / "run_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
